# Phase 0/1 — Setup + EDA (EchoNet-Dynamic, free-Colab)

This notebook does three things:
1. Mounts Google Drive and creates the project folder structure.
2. Loads EchoNet-Dynamic metadata and runs a quick EDA on a **small subsample** (never the full 10,030 videos during dev).
3. Defines a checkpoint-safe helper pattern you will reuse in every later training notebook.

**Before running:** create a free Stanford AIMI account, accept the EchoNet-Dynamic Research Use Agreement, and download the dataset to your own Google Drive (e.g. `MyDrive/echonet_project/data/raw/`). This notebook assumes the data is already there — it does not download it for you (the AIMI portal requires interactive login).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = '/content/drive/MyDrive/echonet_project'
SUBDIRS = [
    'data/raw',            # put the downloaded EchoNet-Dynamic files here
    'data/subsample',       # small dev subset lives here (created below)
    'checkpoints',           # model checkpoints, one subfolder per experiment
    'logs',                  # metrics/config logs, plain text or csv
    'figures',                # exported plots for the paper
    'notebooks',
]
for d in SUBDIRS:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)

print('Project folders ready under:', PROJECT_ROOT)
for d in SUBDIRS:
    print(' -', os.path.join(PROJECT_ROOT, d))

## Checkpoint-safe pattern (use this in every training notebook)

Free Colab sessions disconnect without warning. The rule: **never lose more than one epoch of work.**
Save to Drive (not local `/content`) after every epoch, and always try to resume from the latest checkpoint at the start of a run.

In [ ]:
import torch, glob, json, time

def save_checkpoint(model, optimizer, epoch, metrics, exp_name):
    ckpt_dir = os.path.join(PROJECT_ROOT, 'checkpoints', exp_name)
    os.makedirs(ckpt_dir, exist_ok=True)
    path = os.path.join(ckpt_dir, f'epoch_{epoch:03d}.pt')
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'metrics': metrics,
    }, path)
    # also log metrics as plain text so you can inspect progress without loading torch
    log_path = os.path.join(PROJECT_ROOT, 'logs', f'{exp_name}.jsonl')
    with open(log_path, 'a') as f:
        f.write(json.dumps({'epoch': epoch, 'time': time.time(), **metrics}) + '\n')
    print(f'[checkpoint] saved epoch {epoch} -> {path}')

def load_latest_checkpoint(model, optimizer, exp_name):
    ckpt_dir = os.path.join(PROJECT_ROOT, 'checkpoints', exp_name)
    ckpts = sorted(glob.glob(os.path.join(ckpt_dir, 'epoch_*.pt')))
    if not ckpts:
        print('[checkpoint] none found, starting from scratch')
        return 0
    latest = ckpts[-1]
    state = torch.load(latest, map_location='cpu')
    model.load_state_dict(state['model_state'])
    optimizer.load_state_dict(state['optimizer_state'])
    print(f'[checkpoint] resumed from {latest} (epoch {state["epoch"]})')
    return state['epoch'] + 1

print('Checkpoint helpers defined: save_checkpoint(), load_latest_checkpoint()')

## Build a small dev subsample

Never iterate on code using the full dataset. Pick ~150–300 videos (stratified roughly by EF range) so smoke tests run in seconds/minutes, not hours.

In [ ]:
import pandas as pd

# Path confirmed after running 01_download_echonet.ipynb (note the nested folder from the zip's internal structure)
raw_csv = os.path.join(PROJECT_ROOT, 'data/raw', 'EchoNet-Dynamic', 'EchoNet-Dynamic', 'FileList.csv')
assert os.path.exists(raw_csv), (
    'FileList.csv not found at the expected nested path. Run 01_download_echonet.ipynb first, '
    'then check the printed path from its last cell and adjust raw_csv above if it differs.'
)

df = pd.read_csv(raw_csv)
print('Full dataset:', df.shape)
print(df['Split'].value_counts() if 'Split' in df.columns else df.columns.tolist())

In [ ]:
# Stratified small subsample by EF quartile, for fast iteration
N_PER_SPLIT = 60  # ~180 videos total dev subsample (adjust down if RAM/disk is tight)

ef_col = 'EF' if 'EF' in df.columns else [c for c in df.columns if 'ef' in c.lower()][0]
df['ef_quartile'] = pd.qcut(df[ef_col], 4, labels=False, duplicates='drop')

sub_frames = []
for split_name, split_df in df.groupby('Split') if 'Split' in df.columns else [('all', df)]:
    n = min(N_PER_SPLIT, len(split_df))
    sub_frames.append(split_df.groupby('ef_quartile', group_keys=False).apply(
        lambda g: g.sample(min(len(g), max(1, n // 4)), random_state=42)))

subsample_df = pd.concat(sub_frames).drop_duplicates()
subsample_df.to_csv(os.path.join(PROJECT_ROOT, 'data/subsample', 'subsample_filelist.csv'), index=False)
print('Subsample size:', subsample_df.shape)
print('EF distribution in subsample:')
print(subsample_df[ef_col].describe())

## Quick EDA

Sanity-check EF distribution and class balance before any modeling — this is a Go/No-Go input from Phase 0 of the roadmap (Section 24).

In [ ]:
import matplotlib.pyplot as plt

HFrEF_THRESHOLD = 40  # clinical cutoff for reduced EF, adjust if using a different definition

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df[ef_col].dropna(), bins=30, alpha=0.7, label='full dataset')
axes[0].hist(subsample_df[ef_col].dropna(), bins=30, alpha=0.7, label='dev subsample')
axes[0].axvline(HFrEF_THRESHOLD, color='red', linestyle='--', label=f'HFrEF cutoff ({HFrEF_THRESHOLD})')
axes[0].set_title('EF distribution')
axes[0].set_xlabel('Ejection Fraction (%)')
axes[0].legend()

hfref_rate_full = (df[ef_col] < HFrEF_THRESHOLD).mean()
hfref_rate_sub = (subsample_df[ef_col] < HFrEF_THRESHOLD).mean()
axes[1].bar(['full', 'subsample'], [hfref_rate_full, hfref_rate_sub])
axes[1].set_title('HFrEF positive rate (class balance check)')
axes[1].set_ylabel('fraction with EF < 40')

plt.tight_layout()
fig_path = os.path.join(PROJECT_ROOT, 'figures', 'ef_distribution_eda.png')
plt.savefig(fig_path, dpi=150)
print('Saved figure to', fig_path)
print(f'HFrEF positive rate — full: {hfref_rate_full:.3f}, subsample: {hfref_rate_sub:.3f}')
plt.show()

## Next steps

- If the HFrEF positive rate is very low (likely, since EchoNet-Dynamic is a general clinical population), plan for class imbalance handling in the modeling phase (weighted loss or stratified sampling).
- Once this smoke test runs cleanly end-to-end, move to the next notebook: `01_baseline_supervised.ipynb` (a simple supervised EF regressor on the subsample, trained with the checkpoint helpers above, to establish a working pipeline before adding semi-supervised / quality / uncertainty components).